In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# A Backtracking Solver for CSPs

This notebook contains a variant of the backtracking solver from the notebook `02-Backtracking-Constraint-Solver.ipynb`.  This variant can display every partial assignment that it tries.  Furthermore, it counts the number of partial assignments that have been tested.

We import the types `csp`, `assignment`, and `annotated_constraint` as well as the functions `collect_variables`, `compile`, and `is_consistent` from the notebook `02-Backtracking-Constraint-Solver.ipynb`.  The TypeScript version of this notebook defines these functions once more.  It also allows values that are strings, while our solver only supports integers as values.

In [ ]:
import_notebook "02-Backtracking-Constraint-Solver.ipynb"

The result of the animated solver is a record of type `solve_result` with two fields:
- `steps` is the number of partial assignments that have been tested,
- `solution` is the solution that has been found.

In [ ]:
type solve_result = {
  steps    : int;
  solution : assignment;
}

The function `is_complete` takes two arguments:
- `assignment` is a partial variable assignment,
- `variables` is the list of all variables.

It returns `true` if every variable has a value in `assignment`.

In [ ]:
let is_complete (assignment : assignment) (variables : string list) : bool =
  List.for_all (fun v -> RMap.mem v assignment) variables

## The Backtracking Solver with Animation

The function `backtrack_search_animated` takes six arguments:
- `assignment` is a partial variable assignment that is represented as a map.  Initially, this assignment will be the empty map.  Every recursive call of `backtrack_search_animated` adds the assignment of one variable to the given assignment.  The important invariant of recursive calls is that `assignment` is *consistent*, i.e. all constraints $f$ that contain only variables from the set $\mathtt{dom}(\mathtt{assignment})$ are satisfied.
- `variables` is the list of all variables,
- `values` is the list of all values,
- `constraints` is the list of annotated constraints,
- `steps` is a reference that counts the number of tested partial assignments,
- `on_update` is either `None` or `Some show`, where `show` is a function that displays a partial assignment.

The function tries to find a solution by recursively augmenting `assignment`.  It works like the function `backtrack_search` from the notebook `02-Backtracking-Constraint-Solver.ipynb`, but every partial assignment is displayed using the function `show`, and the number of tested assignments is counted.  `RMap.keys assignment` is the set of the variables that have a value.

In [ ]:
let rec backtrack_search_animated (assignment : assignment) (variables : string list)
                                  (values : int list) (constraints : annotated_constraint list)
                                  (steps : int ref) (on_update : (assignment -> unit) option)
                                  : assignment option =
  (match on_update with Some show -> show assignment | None -> ());
  if is_complete assignment variables then
    Some assignment
  else
    let unassigned_var = List.find (fun v -> not (RMap.mem v assignment)) variables in
    List.find_map
      (fun value ->
         incr steps;
         if is_consistent unassigned_var value assignment (RMap.keys assignment) constraints then
           backtrack_search_animated (RMap.add unassigned_var value assignment)
                                     variables values constraints steps on_update
         else
           None)
      values

The function `solve_animated` takes two arguments:
- `on_update` is an *optional argument*.  If it is given, it is a function that displays a partial assignment.  It is called for every partial assignment that is tried.
- `csp` is a constraint satisfaction problem.

It solves `csp` via backtracking and prints the number of tested partial assignments.  The result is `Some r`, where `r` is a record of type `solve_result`, if a solution has been found.  Otherwise, the result is `None`.

**Implementation:** A parameter of the form `?on_update` is an optional argument without a default value.  Inside the function, its value has the type `(assignment -> unit) option`: It is `None` if the argument is missing.  A caller passes the argument as `~on_update:show`.  The main purpose of the function `solve_animated` is to annotate and compile the constraints as in `02-Backtracking-Constraint-Solver.ipynb`.

In [ ]:
let solve_animated ?on_update (csp : csp) : solve_result option =
  let annotate f =
    let vars = collect_variables f csp.variables in
    { formula = f; vars = vars; holds = compile f vars }
  in
  let annotated_constraints = List.map annotate csp.constraints in
  let steps = ref 0 in
  let solution =
    backtrack_search_animated RMap.empty csp.variables csp.values annotated_constraints
                              steps on_update
  in
  Printf.printf "Tested %d partial assignments\n" !steps;
  match solution with
  | Some s -> Some { steps = !steps; solution = s }
  | None   -> None

In [ ]:
solve_animated { variables = ["x"; "y"]; values = [1; 2; 3]; constraints = ["x < y"; "x + y = 5"] }